# Cuaderno de optimización del modelo final
## Configuración del cuaderno


In [ ]:
#Instalación de paquetes
!pip install tf_keras tensorflow numpy matplotlib -q
!pip install coral-ordinal

import os

# Forzar uso de Keras 2 para evitar problemas de compatibilidad con STM32Cube.AI
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import json
import shutil
import numpy as np
import tensorflow as tf
import tf_keras as keras
from tf_keras import layers, Model
import coral_ordinal as coral
from PIL import Image

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (
    cohen_kappa_score,
    classification_report,
    confusion_matrix,
)

# Verificar que estamos en Keras 2
assert int(keras.__version__.split('.')[0]) == 2, "No se está usando la versión de Keras2"
print("Usando Keras2")

from google.colab import drive
drive.mount('/content/drive')

import sys

SRC_PATH = "/content/drive/MyDrive/TFG/src"

if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

In [ ]:
# Macros

DATASET_ORI_PATH = '/content/drive/MyDrive/TFG/dt_ori'
MODEL_PATH = '/content/drive/MyDrive/TFG/entreno_final'

os.makedirs(MODEL_PATH,   exist_ok=True)

IMAGE_SIZE = (480, 270)
INPUT_SHAPE = (224, 224, 3)

NUM_BATCHES = 32
NUM_EPOCH = 5

SEMAFOROS = np.array(['Benidorm', 'Daroca', 'Delicias', 'FrayLuis', 'Martires', 'MCerralbo', 'Pardinas'])
SEMAFOROS_TEST = np.array (['PJesusO'])

LABELS = {'0':'fluido', '1' : 'moderado', '2' : 'denso', '3' : 'saturado'}
NUM_CLASES = len(LABELS)


## Funciones auxiliares

In [ ]:
from preprocesado import ImageCropY, preprocesado
preprocesado = preprocesado(IMAGE_SIZE)
from dataset import build_dataset
from evaluacion import evaluar_modelo_tflite

## Carga de datasets

In [ ]:
# Carga del dataset
print('Carga de los dataset: ')
ds_train = build_dataset(SEMAFOROS, DATASET_ORI_PATH, LABELS, IMAGE_SIZE, NUM_BATCHES, train=True)

ds_train = ds_train.map(lambda x,y : (preprocesado(x),y),
                        num_parallel_calls=tf.data.AUTOTUNE)

# Dataset representativo
def representative_dataset():

  c = 0

  for images, _ in ds_train:
    for image in images:
      image = tf.expand_dims(image, 0)
      yield [image]
      c += 1

      if c >= 300:
        return


## Optimización

In [ ]:
model = keras.models.load_model (f'{MODEL_PATH}/final_model_ft.keras')

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()

with open(f'{MODEL_PATH}/final_model_int8.tflite', 'wb') as f:
    f.write(tflite_model)

print ('Modelo INT8 guardado')

## Test sobre el modelo cuantizado

In [ ]:
# Dataset de test

ds_test = build_dataset(SEMAFOROS_TEST, DATASET_ORI_PATH, LABELS, IMAGE_SIZE, NUM_BATCHES, train=False)

ds_test = ds_test.map(lambda x,y : (preprocesado(x),y),
                        num_parallel_calls=tf.data.AUTOTUNE)


In [ ]:
RESULTADOS_PATH = f'{MODEL_PATH}/RESULTADOS_INT8'
os.makedirs(RESULTADOS_PATH, exist_ok=True)

evaluar_modelo_tflite(
    model_path=f'{MODEL_PATH}/final_model_int8.tflite',
    test_ds=ds_test,
    labels=LABELS,
    output_path=RESULTADOS_PATH
)